# 🌿 HUẤN LUYỆN TỪ SỐ 0 — 3 RANDOM SEEDS TỰ ĐỘNG (`99, 123, 999`) | 90 LOÀI + BỔ SUNG THỰC TẾ

**Dành cho Google Colab Free (T4 GPU):**
1. Bật GPU (`Runtime -> Change runtime type -> T4 GPU`) rồi nhấn **Run all** (`Ctrl+F9`).
2. Hệ thống **KHÔNG đọc bất kỳ model hay số liệu cũ nào**, khởi tạo hoàn toàn từ đầu (`weights='imagenet'`) và **chạy tự động tuần tự 3 Seed `99 -> 123 -> 999`**.
3. Gộp sạch **25.013 ảnh gốc + 212 ảnh bổ sung thực tế của 8 loài trong 90 loài = 25.225 ảnh duy nhất** (chia block/group không rò rỉ frame clip giữa Train/Val/Test).
4. **Train tới khi bão hòa thật sự:** Pha Head (`max 50 epochs`, `EarlyStopping patience=7`) + Pha Fine-tuning sâu 50 lớp cuối (`max 80 epochs`, `ReduceLROnPlateau patience=3`, `EarlyStopping patience=8`, đóng băng BatchNorm).
5. **Tự động lưu & nối tiếp từng Epoch lên Google Drive:** nếu Colab Free ngắt giữa chừng, chỉ cần bấm **Run all** lại để chạy tiếp đúng epoch đang dở.
6. Cuối cùng tự động tổng hợp **Mean ± Std**, **đối chiếu trực tiếp với Báo cáo mẫu**, và xuất **bảng chi tiết từng loài cho đủ 90 loài cây**.

In [ ]:
# ==============================================================================
# === BƯỚC 1: KẾT NỐI DRIVE, CẤU HÌNH 3 SEEDS [99, 123, 999] & MODULE LÕI ===
# ==============================================================================
import os, re, json, math, time, zlib, base64, random, shutil, zipfile, tempfile
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from google.colab import drive

if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive')

# --- CẤU HÌNH CHẠY TỰ ĐỘNG ---
SEEDS_TO_RUN = [99, 123, 999]
BATCH_SIZE = 32
IMAGE_SIZE = (224, 224)
LABEL_SMOOTHING = 0.05
MAX_EPOCHS_HEAD = 50
MAX_EPOCHS_FINE = 80
UNFREEZE_LAYERS = 50
LR_HEAD_INIT = 1e-3
LR_FINE_INIT = 5e-5

PROJECT = Path('/content/drive/MyDrive/KiengLa_AI')
assert PROJECT.exists(), f'Không tìm thấy thư mục {PROJECT} trên Google Drive!'
EXP_ROOT = PROJECT / 'Thu_Nghiem_Tu_So_0_90_Loai'
SEEDS_OUT_ROOT = EXP_ROOT / 'seeds_mobilenetv2'
CKPT_ARCHIVE_DIR = EXP_ROOT / '_live_ckpt'
COMPLETED_BUNDLES_DIR = EXP_ROOT / '_completed_bundles'
for _d in (SEEDS_OUT_ROOT, CKPT_ARCHIVE_DIR, COMPLETED_BUNDLES_DIR):
    _d.mkdir(parents=True, exist_ok=True)

BASELINE_B64 = 'eNrtfd2P3NiV33uA/A+EgAC9wEjg/SB5b95G0mg09kjbUAsz8QRB41bV7S7OsMgyyWpNzcIPRh6CIFjYg4WxCIIFZmwYhpMYayf7EgnBAumJ/4/+T3Iu64tk8X5xSlo7SXepqtTNrjp177m/833OX/3zfxYE92aympbpsk6L/N6/DO5dFEGWylUwK9JgOlfP6lUwFdNgIgp4LIKFWAXw8zyYFkFdFvl1MFsFIg+eZEUpHha1yNNp8P5HwVldropp8MU8hctEmgeZSNVrVUUQ/sW995r3zsREZtXlTQrv/K/VT4LgrzYP8MtpJqrqMp3B78L39j8tyvQ6zUV22fytovj9qzKdAgGfpEUm6+DsAt6sXopVlooAPpTI67nYvmHzCjeprHOxkJW8VA/qJbZ/Op3f/nIVnM/Te5uLf/KegSZkpCkrZPCJLIX5fZ/PRfDd12Lh8n7Y+H7wIVdluloEZ4enIp+VQop8tbB8/Kd3b/4G9nFx+w+5CyXESEkppyI4FxmQ8ni9rNIqyFY18JjMKwsZj4Czbm6/ya9diKBmIsri1VyKWXCewf4DT6zz6yJXi7IsZsVyvs4y66q8LO9e/2YVTG6/aRgjD27u3vyH2oW2yEhbtRSluF5VwRNZ5rBf+/9XshZTubIt08t5evuf82a3gHng/jqY3r3+VT53IS02kfZQTNbBi9VkIsvdwp3LpSyLBRylYlKvqvSqgGNlXbg//v7uzd8BXVWaz4Ps9ltAgNv/5sRbiZW+l1KUFewonNi0VNQAomyfpjYGu/3lOsgVcf9pGizuXv99HcgFbPBvXShjRsrSchYUV8D3pZillQT66lJmaf0V0FdKdbV029bvvk7v3vzblQtF3EZRFTyXVX3gs0w2RyBPZ1Yme/zH3//xW9jBL+/e/PugvnvzCyUMnHAKGcH6YSamXwAyZjeAt4CSU1HBCp3tnzVrtcpsLPZMMX8+XwGF330tnTgLITNdq2qewss9BG6HjQPoOHsuCyBH/QcEn+Iv+yY+vnvzW7E9jsHh753oMwL8w6Kqi3y7lc/lcg70yCWAq/xSZLWohc9+bl7MiSgj1j8SWSaCj9NsHZx9BnAhlS4xh+USsp6nxRLEsk3yFLDlKze+omZSgNE/AnVkB1zA7imcRFA9ApkJ0G5K2/m7e/OzNCjv3vwXNwGEjCj/CJhJBI+LLNsR9ELMZLkQ0zloBQoY7YvTwNX87vW3wEvL+e3rqRNZsY0seIvggxtZXpcStLiLVCil4ToTRS4XwLvFYrEChrKKx+fXSiYCeb9bNS/iRFziQtzTdJJWUxCIZ/tnZVGJ+7BooESkVui6/SWA+t2bXzvJasRcSHoGi7Pey8Q0kyJYbiVjkQLX23YSeAuAaqNm1QDwPwU+Va/iRCE3U1gCly9EFTwS01ot2sV0DsoNyPBrxWmTMp1dy8oqG//VFiDKDZEfprffbgW4k3IaWmhcV0oPlws461ZAmDqyOjYi+iMASLUc20dgo1oCXJVOG/by9nciqG//wUOzwthMTSky2JMSoPL9EoBJCZVS5nbk/mEKErs5a5nSXH7lhk6YWKjZA+XjUkwFEBJcgRZaitxB/bx7/Y816E9v/uMUjDonOYLN4A1ItFTC90l6HZw9SdWJX64WqVUN+AR2Zx3cvfnajV/MiA0mQ650uJdzWB1gnA9WS3g2gY0CSnzPz+T22xpOOmh0TpSZQbuWjRUrzRR89/PV7T+A0d5YBIe/cXl7Iyw/FldXxSzNYEmei3KaVpU6TsvlAwfZXs/v3vxm7bEQRjQGVgUrLngp0gy2Z5kuS7nIlU2X5rmDxGoZdN993azVZrOcCDOC8GPAW8DfHPT+x6m8upL5RDSKkMMyfaIEPOjkSvmv5wJkfeclnAxyI/p+kF9naTUPProB7HkKOhpIhTlopV9aD/rOjPtyBfDjRIgRkZ+IfOMYuB98LGbrrZPgxVxs9NipzCq7aQmyAUzfX66dyDFC8pN0Nstk8LEUV23gydalHZQfKjcFqD35tViCku9EjBGRn4AsmM6Dp+sZQPA1qBdnh6egNJYbn4VVY7x7/dsFyK3XSm79xsmIJEZk/lCUM1ASARuV9QGydJYKCSfuBuSovHY4c49AhtZg+zsdMmIE6A9TJa9egiXWshebtSnLefpVYRNcL26/XQdfACD9ArYug5PvRFJsJimvZCO3GnV187BIgSQwGJe23bpQh31S5JVwcjgSI0p/WGSz4PEKrKC9QD+I9mpVTldZ4Xi8lKL6tZM+T4xw/VSkoO6cz9OsACaaKePsrPO/6scrsUivZOnli1sBdWoHlZxzItII3U+lKGsl9n8gq0ZdFJmYKS/JJJ0WmdVqbBwRysO1uP3G6bzR0EpNphDJsGzTopz5ibvGBefmvqFGCH+6FtMU1AoFT5tnsGZwmcxrkdktMxB1SmMM5htlyYkeI4Z/lAMg5cH7XwllkoEsKWYH7koXdrPnu5/fvfnb4MertaOCQo0o3jggACyrKphI5XAGmh5un4CJph5z6xmcr5RV/asUzuFOAE8Btf5eKVOvf7d0otII6j8QMxksN9r/oxL+aJWJoLixS73n13dv/noKHPW7qZNJRI14/kM4aSDzCungBak2Z14h0985gSU14nbjsoIzD/oxiLIsA9seZFuunoJgU3v1uQM7f9x4t90MIGoE72diJq5FBUIj2Om4YBK1AHwhSvUn1h06Bxb5dR3M7t78Wjm4HU1XaoTxZ/LLJr52kRevJrBCoHpP5/JGqoWSGWgBVpPxQjaRim+Doz90os6I3xvPzGa5zpXGDNqtALlS2/1re8vaKaYTWqioaqVYPwbFepqCpA3O9j+b7X7mB9hf3r1+E+xexIlEZCaxklWQ5g3Tw7FXuu/ZS3iUwHd53RgrAgyQqbTR+bEibKLMFieqjAD+XJY3cueKeJLWdYOVIpukNzJPKyd9t4kDOJFiBO/n6fW8vq8ClbWcbb3aez0T5EopXBTMev7Hb50EbWTE6L8sgZVndgPbEX4iIxKfixI0na099mguFkLCx5Wux1vFbesmaqXifL90+/SxhSCwOFRKwStZKqMVDlDDGMsKWESAAmQ3POZKT0yVm+pnwRe3v3UL3BpR+lzC4QgyYAzL+QCwvP2fTu/HzO+3XMpykaqj8UyqNIZgmcJPUrt5CodzGsxvv3Eiwoiv54WycWqFDy2P2HKVqUBGmS5cQE35xfJrZx0rDs0EZV+oyEq9Q42n62UhASSroLGRiwoYZL52Cq5MlbhUqr4TWchMVr6ulWNqc4oeSrECcZ5LFdKcrkoH7WpLUZNpULhRZITW81KsD4H8Z6KERwAMuYKjJFdWNHusUntu7l7/IQ+2f+tEkhFiX4jJJIVzXRS7qPRjoZQu4Kgr+bl0iu60A5mb1VJOxp85EUfNxKXKcFgGh5yHVvrDssjWk6Iu157ZD9cAjH9d77IOnKiMzFTWYCBWufhiLzaVyQpCvREGU6cMjY6rWOkas9tv3DjOCNsv5Oy+zXgFY3/ilA3U8da6brARv1+ABnR/qbLDQL7vF+2wfGC3SRW0rr2Wb+7sQ46ZmbpVvl2+AdoU7I7aWefMm9goBrpZQRtHl4Q/dlCzv/u5gF18vXTfxsQoAC7AQNrpKmvQzgBgb4psVQsnR3u9unv9j05EGOH+AsTf1VVmz/C7/t+/BQxQK/Bfnd7VCOkXaXajdqGo50UT6QXEEkvlt1Jsu/I4Ukp9d0KjxAjoFw0ObX0JF6AqSnhbq2j5WIHht6k6Or9wosGI2xdKUIEx3/iCWjZzKWF/vnRzeU5uv53OFT2/cvKvJEaIvlims8NZeTQH3Vq57+vVQmVXFJVVd33c5KkBNf/OLWfOCMoXtbhWEc2twAWq6vVUlhsX59WqnDq4EjtCt2pSZDY+PCeXZ5KY6SvTzQaCfl2qKOOFzCXYy4DHrzK5FvnK2b8wVUr/36agbCpHlRNtRky+eJVWVfBoLlUOyCepiu/trXoxA3YHoyT1sOqBOCeijFDc+DoMEnaSbuKh6VU683ESb4KPzrE+ZgTpl6ssXTrEh5vLXN7NiMafyFzl02bruhRLBy170uR+lsAsf+Ok9jAjKn8qgCEWMlPJZ0Oqoyiv16X0TZud/fH3QjmCf+10xpgRpj89zq+YyPxzAbalNUjltkBGhP50ntYyMObI5iraIlKvHFnQbWAf3XQvZkTsH8lXANeAiOfNGT8vZk0ATwXz9rFXu3S9/R9K54JNu3XzjDEjbv9oNZ0KB17eXOfydkYY/uyzncD6DNiimIKBD6rVV/DcSe9UvtRNYpsTKUbU/UxOAF+fildFWaskirPd06IpJwADCI673Z7eS4XF7X/fJ+Y4Ucft1O2yTJdz0IJVnUUTzCxLu5d3k4QOAuoPYmteH15kR516+DebEplqtVxmctE4JS/ZZUOmrA71Ms6lHz6FB44JYqN8ylad0Vmh81G0LBDYXnApZ9XlRFQATbnasS2P3OP88B/4L6zErFhcqstVLn6LZ6olgNplqZKO1Yaz8D5StxZb1LKqL9OFuG52EtOwhU7ToizltL5clnIGSn0K+kaTOdtO8knz4avitmJaF0t0KabTFazfWhVRPUgQJ4Tu72nnWtK9ljFMMKO7+9a1DSSqN56m1aZoDF6ZhBHhPOQRS1jSdu5urlZFQVm2uTQMY8IiElKSRFE7vLu59AptL8MYw6Uhwwmnbah8JZWjHk5DjwQawsvyJEY4THA76rD/gzYV2qXYX70jBAHFmPCYEgKEt/SQGWAQHJ3LuqhF1uRcsnaUKC/qjTO2CKYr+OUD+G2gVKwzUFNVonUwKYJKlTDAqZ0GtfyLHjjdQ5h0WG73hmADbj/2PRI+CMMw2PBScLZ8Kqp5MM3g6MtZG4i8eTJqJyroeJIndp5M2iHzAZ4ME8pCgiiPwxi3j9EAT8ZJguKIRLAVYRIyC0/CZ2DAaJiEwJcJM/BkzAnHIYlgfxnwHNLwZAwcjnEUIpwAK3Bi50k4FvChMEnCmOJ2xcAgT2qX4ognQ8ZIHGMWRyHH7aTJI54EaskgT+LoAfyqYcgN1+zZEiMcbFKTzOzJe4i4h0L+3j8Ny7YNTQ3LsnYZiZZlYwvLEo44BrDhiMbYzLIRoAsDFogAUBmycizGBCFAHOAYbkLRmDU4RwiCk8BjquVYFCMCt5DSKGqrXVqOxRTFwK5xiDlqJwYPc6xuJY45ltLmUwGDR2HbnnqXHEuxQYS3RdHbEeHtcJ9WhLczRQZ4j4QcQHB7j8wSXCHJ/j6ySXAOIEhgPxECzIyMEpwnPAqThIHg1AtwhAhgfwQ8ymNOuZ31ItA0QhRhCmQkmFpYT7cQR5yH4XhwOIXwJGHtQPhI+X0kpcPQwFXw27fNVm3PvZat2omsQ2wVRSBK9vdmvopp62bjKxolFIM2x2kMihwy8RWckEghNGAKihnSMlbCeBQnSClunGMHxgJADxMGApOGmBIbY+mW4oizCI1AHQljeH0KKH8yzlIPG/a6V8olmL7wnoqdLq/LYrWsWjYJxZegJF72JfHLYnkfBe8fdm3/q62YvmyQUH+KdpdtNFCDarK7cEOBQSKoXQQRv91BinAE4lq9L21JZPVqddPwAZAe5AUOwZjANIb1wbT/UpdwKfwr1Uom6AFFwf/6QwCPyb+4t7vyJ+91loS4LIkWNvtrotVG+2ui0wHaS8LgvWDpYIkTnFAaDS8JRvBaYLCAPRaBRchNS8Ji+CibJcF4cEk+3fJzcN46LlpO0aHzEavodN4jVtGpGh1WIQw4LuIUg20bM65ZF1BxKKhOEU5w2GOoI1aB4xZv14Wa1+XFDhX+//HZr8kTdP9iqkpeDKuik7jHy6IxY46XRaM99paFIMZDsMwosA0bXhbAbABtAhpJiGPObatCNqtCotaqHCmVQOdlXws4GQTvlkDnsNgvakOAQX511grWHMOab++H4SaEzYGXwMBHlOCEmVkIb48VYBl9ywi8w1Wdi6q/JFqNpYPAjFAAtt19PLwkAEMEYxyDwAfuQSg2QjB7QOhmTXDyDiB4xyo6T9gRq+gUow6rAHgw5RkB0AEcIxpW4SGobLECa4oQxkZWATuObFmFv2UE/r/m9JwUgPeronFtHq+KRtPtrkqiJH8IXMUBgZPhVeEh2DRwXUQYTmIcWVYlQZtV4eExALfU5ImqnlyI8otLerkoZqqp2IAHf7qqarDMpnnnlN1r/uDyC6kwqX3Ne70r6rTONiGc5prg0fPnwdmTEp5dTMGmm87b3isVW1lUlwtloT1oO16q9Ct5uZg0/QsetIyyTNTwIdaXi8bwe0CPzbWt4wfUIASntZGIaMCHv9M5IxYTCoopHO52hkjPNgoBDxHIRKo0iXYPmo6zSfeeOzeT4jICJh5nYJq1CxVhoZZL4CQwY6bzXr7FRNm6+1+EvV/ciOzwoTlPelJ3UUzSTObKtYj1u9m5Srufz5qrnsv6ExycqYqwnwb17Tdpk7D20+DLlaqp0ewtftD5tIe9xQ9Czd5G2r21Oxa3e7v9PexYSNvJUX13IoXfoxhU8aSt6txz8uAd4jCw8SAkOAG1qd1EqenqB4u4gI+c5k068gNkcs72drCUFWxMFOp3b3+FdudeyOq5uqKJOubVlSxVPXeZp/m1bsPIg2R4x0LAYM2W8Qdct2VUQTxHgHkgLZj2OMJZDBHISOUQa5+B3o7BWQQdFuQ86KksGj6N+rfcbRmcUhA+GCmJ2HH8HW0ZTR7EvW2RV1fpVBV0qlMzMexO/0LtJn2wuxC26qHHVtEH7WZGrZ1i7e3obBQK28euu1MxSzAhGPQ2BCjFNDvFQ2VuAtyxMATLHmu3KlFni4LU5CRO2l2E2lulfc/dVsVgqhASgSxULiJu2KqI7beqJfiWstzkBAzJO7fOmO0N7vqRjj40J0lHYLc+qlLGIwVJSH2msKMXbz8sDzvfXXVjtVSOrm5DgoPPs2lyMaAV9SKkxwSHPRWjTXGLXOWZGaBYBbNb38yPYswHKOa2JW5RBAvKtMsdgb2L9/dDxHMQymAA0JgAHyKsIT7SLDcaIL5R0PYvDzoD6IQUJRiBfdCDkUMHVA/+SkhnT7jmw/MYgfpId/dDvMZI92v4w+NYs3PRCF5juL0hXbdMh+1Q+8YHiY873xriEw3xZATbsZiDTo6VU1EFC4mGeAC9zm2IeE9qQxufgXnf/u7zmS39yYP9FFgDqmPQZRGoQzr2Y5slUm7YGNTnwR1kSHkiQeaDOYS7lxzWhGrYj6IxUNdleKw7O13aB3EakxAUxt099iQej0K9uPOlY789fzYfYGjhQwpAvb/zXHhCbMwYc9h9lSMSgYHNj0DP2PzYgxNjLe4r32bMKEYh8BUZEloJGOLt7+El6ALPYQlQMoL3Yp2MPVAK6ipIjAFy41i/9Q7kjgE7rRJDhwiMeipEC9PCYaK4hYviWDlm9t/siIvculd78JP2Aw98Xtr90n12jd6AxqhpuvNOho64L0VjtLC2KASo0YnDqAWTYTIoCWKEwQxnKmMCgIMkfooIiiysFKk0SQ5gx0IERiHts5JDzu+fmP6PEo+lsDFWouOsQShiPdb3O/VoDKPpcD6JEQ/p4X4I57uyTqd3IQ21sU3UURonYN/GiMDrh32Qcu8V7wNTXS5CPqjVXTH8DlCLdFlei7FxxywgQ8QjTiLKCFNZf0miYzx2QhkY0e63ToB3aMdDfBip0A/hhFIcAxZFnsQzCx9SHmGiwm3w+ggfeREdZgL4GKDaTTRg124duu4E3SZidylltTh1xLIhQeR5Isao8SbttM1Ubes9IoPgFnW/PMVmYtPAMCNKLgOuAVv3XaBu0xx82KqrkROtDO1+Da0L735pZCjzAH2rDI3Gm5I9hNTJKKLxDaAxwNYTIrq13hC9d3qMtkx0xCObFZDEhHAgATMU0iiJBpjQPMDDhwF7Brb2YEYowRQkPicoJENyNkFgAiNEVJwXJLRuUbjHolhhjutlUFtlYghHdlegBpcJ9nAFWnHQ8Jbt5R6kN0F6fcGBXps/DbaNKQM0jmLQOI6MT/cBLc7shx6E4wGvI1NJRN4B4BmsnGQ8g2nNhlEMZrC2Ov4yq+qZ9GS2pxVhs09ZROMYoRgnKATgoEfc5jp2x8d5pnW9D5627pfn5o3hL8fNUzHI9vcQ8V33A/FTOUdxntaiZt2YxbBzlkVIVX8qj0WCfcm1C1SwPkLANOC5iNG+c9ZhhJJPrKCrG2jVjL1N15h4g7GCCNO2DaXBe80JxKM4sE0TI1qJ2gPfQexrhwpC3whpMgr79KFPzwip4ZJTREijmCCKEpXmBiYP6vGj0/Ss0+t4STwYMwDNT2Uqhgz4EPvaivyUZkUyGNQgFPCN7en0JJCdMkrwrgjsYBsF0ASlnIP5yDg65iWH8Wc+IvQQs2m0g/Gx7CTsCqkTxLKtfjmOWOs2jG49pXQf6wG7J064UlmikDI/62ZUwkeEMOKtOy3THcwy4AUfDw/3sK67ThIScZ6ECINxH9J+5N1xxN0JolRD5w33sNszREfHIJab4BkMcnaVTOyp+Yzxg1C9FUO6iuXQ8pJe5NaPYJv7DV4aAA1EDQUZGbE4GmAt9zGFXrjWSXTRBRiiIRkZMSczT4fwYzguYmGsMu8R5iHBOB7vpI8RKJicchThiGLCTuCktwJb4pQPRXsneSg8sg+fxo2Y9SQe22KosDicR0zl6gDfa3jRPpXSx3roHC6tkzIe9MB1ncm+ka5RLmCDnenpbjAkkNm3ctC5ZbUXDJ7e9lp34X1QS6ZRyFWZrkp4ZFpfCRtrv5IkCQkjUUJJGCGqYUT7LFKfiD6Ynu2bNu1ItXLY30eDWWMdjtbIY6JRRsiojMsEqd5Qu/vvkTPFGAGN73CvIV6zs4SOSXmjHfJ1xPOkI4oHXaSAX4Tt7z1pt8lqrk5NDO8cqUAP4Udc6Tx/1seWNSCJg8s/oTGGA6qaRMRggLMTuPytEEndghSt3G7lXRkMfRLQvBFjNIwSnnimyQ/6g6wQibRZ8i4E99LkT0Bwz60X8RAjFIYhqF3kmAFbw4W9oE8fWfDLxDFJ85GZOPbChs431jpl9Wnde+jTx2AcQrSjJDKinZs2vqyPv+7zcXFbKOtWXsd62JruS9URjMIQK3/vkUS2jJj2UQgdMyfY6IRfrUKYjKp10Mf1bXaUY1jdx47ipxEnLumc++BFUygQnzidE945UnEAVbqZgCJyxHCGKeJe5ke3gsbHk4H1avJIT4aV2bSyycD4+5CTSh873HuSOyo+4RYZino5xkNrbcied0hAsRUzqK40RPkDAD3jJOLHzGbqAXyK7HM23tj3SVuzOoy98s9J/A7yz7s6h88B3RYvMQ5KexJi5IlQNqahcaLahQBQqeqmY5FYHvdZXq4WaeYJUO86s9HqiUPdWIeX45B08fYdCLwT+pfICfxL3agCsCeIjt39EQMVr3KVfPlSDS2p2kPngIt8zUhDckt7GQaxt+Mk94UhfxNSnyUVh139ZXy6LDuhmOv6JRLuA1E8SiiN9+kgntRSG4clSRLGiHAaMhwnR3GrWjal79LLWOxmy8cenGRaKHuWAR6lMRkCPO29MWSs79bS4Naw2/bj4u6GSGd7qVHXNTHa0tA6JqzxUYZVxDZBkTKyo37g4LG4uipmqZpo9FyU8AHVnK5quXzg56A1FB60bWUSYoKBCGD7iAzmGXEETBxizBmLQm3AlGq0SRqOctBqc8eTGCAAsTikcYTQ9yhDJqcsQzYUEXWKxwgF4yzB4WbZB32yHbeZL/E21uOqjQxVXblCypMjp+xuiLiaR3r2wTKFT7nIVfloMxjKt3a0mxOuT1DQqw170d+JoHgGlUf5yUzFV20kpMjmaor8fEujirIo637roC+2OsaiGNGmV2aSII458yQ+toZMMY9Uf0Y1MAGzI/5bLSbBI6FmKT1O5dWVzCeiSW3zhj9GCXAe2Kg8RDwOmT4QoNXWdp6AVtGCqmAYXpFII4YpGZUw4qR+mvLH98SH7Zvm5ESa7aSjUsyp1uvS+WKDXpeOJy3xJNfqlg25wlSO1ACE+Ch75IP8OkurefDRzVrNDp+pMNRcZumXnlnkh8QLlYcxfvMcA/bUw8Nu5byQORWg2AvxYnaC0ju7LaHv1tNea0PcYO8Lb+AO0y0AeqYb2jgvwbGqGYzAvECqq1KP856IfNP9437wsZitt51AXszFJs18KrPKMyPOVHFvT7hveia3bidIuLenLDmVPrFBfZ11c+H9bO5x1YI6+gwItk+p0hsgDi2mrNZFrHKSQF9iHN7kyDv7JJ2paWsfq3nRLV9bti59gwE9NqJerq2uJc/fRU6cqTyrW40wuGuGEkcHdOOjMjGd0G0w8y12qQUc7cyNVEMlqtJ7qere2mexUnXVDZ6uZ6XIr9U48sPTw1xMzyIFR8E06Odt59eHjLyLOGekd735KeSJpyg6bcmCC4Fhp3D6FOR2JCdSib4xoUrniFFfZ/tQlDOZB4/KQtVonT0qZqmQYLHeiDKV1942q2OqNqWDphMFDjvce7IZOmUjB3tu7N7ls3VInCA39kTBKeW8U02ocRgzHkeDxGPP8nEroDVtrEFlB9kJC9aPn3+YqmjmS1EWrYrmBsrKcp5+VXgW1Rsyg2xBll5eTvQOkngdO74MqtWdsk3qXd2E32J8YTB4Y1BpRqolHdVMBa5U0DykPE7IMZfllWxCoE26+OZhkQKXqaHSfvKyjwCRV6S606Ej+f5haxuLYUMiITHknOzo3br8t6jB/bSdccVXbmFdQwR3D2WMhIixOOFgwmBfq9nGcjTknMGJSNQAEIbxkfwsslnweFXVwT5V45C0Ua3K6SorPG1PfefkYTnU+Yo96+jHODm0rmgD3+8bbu0jDzjh2jIEckJOG08r4Wo4y/7u+9PaYSzVmEFNfFTOUYCzHl89FWm5Ds7naVaAfjZTFaZnnf+pseeL9EqWvr123VqkmNhqOMqXeHYIOW1cywQoewt0y3VxEwTzVbTp22uv26vN+R7KceKhHHeTQ5KmABd0DazgtM+NUpS1Sg75gayaxFuRiZnqiDRJp0XmWeOc+FXIv5OaeMeyB3sXbscknuSEWd0mZ6NfnVXcC3qfus4KkLb1jYaYLFN+NQPswdvN/KtPDe52q/Pj+2VNj3F+RMSt3tmeNe2oHMSn7FOjdYX08goGD48ngTbnR0y7R7bPcWsxTfN6rhxsm2cAbkWpRqaIzLOydB+v2oavdP7t4U5WTrUg2ropNqaQJcbtm7aGT9PbjSESRRw+ckgI474Ex29vQIcapt36poMJLhEBcUcoAdshYroBHWT0gA4GGEMQowkBe7+v2X2Uz1KRB+9/JVQF6dmLeTE7KHXpwrtiz61EeNh0arcaoe+ida8phu2pzeHu1zvQ5kzNJPy0Odo97vzE2hxwHQsZj2Pl8D1u/tb0q3lUFlUVTKRqcA9s+HD7ZCEq9Zifps8gnLTObbiItiOdPRWmMSWjXRegNtun7UNFyWCWpqmex6ERzKhWv4bGZp3a607AbbCEz8V809JubWPOEY0w39/3ePAHYiaD5abI5VEpqmqViaC48Y6h6gsw9Wd1r22E7exNXZZq7CEG7P0c9EW9nZQ+PbbtK62d8oPjEwrgZHxekmMqduyh33QbOKhh8CgOMY4Rjfutyn8I1mo+nRdeiee9UVDaSSvEnrDQC0Pr0kEQOWHLJENQ0daYUwPB6JRBBpNh0mmYNKh9alxxSMM/kbXTfXf8RF9ja1oFFldBPZfBDZAm1ypQmqunomxk5ufe5oKj2tYtohw0mrqtPXWpIBSfMr3cMIirm2uuzere51Pre606ED8m84gbCmk7qovVXcKdSnJ1xBObuwQsSoYSEE2EYzWYtMeVz8RMXItqKspgl3ZeStmKPixEqTrm+0rU0CnzwpRPsve36r0uDqnQozwn0ffQyeNum9wT6ORWg2J81kjXckMnThpp8t8IDcFQxqBF9nPgnskvm/mmF3nxagJEB2cfTOfyRiowlJm89q1TNiX92XIXHScdnLLKyzHJhdkDr6bEWodUxrc48c8h5dexB8jYlF/AF5UfRxV0AKv3QxKbblwbyDtXRQ6lCMSPV6L2bX4ZI306r60DfZxQh6kgWr02+X/NhOiWb0V65tNXXew9KJ0xXLquZ4nHTOOO4xhWF6t4GDwBRBpgvqpW5Q2PZZZO06ISwdn+Z7Pdz/wSTtQ465AxkL8Rp1xbBOIwD4nhkPEIPoAqEor96m3IqNxzQ2awvUAo7oavTlEgZDc/OhW42rqHXurWIALom206EG+dl6SCsVEYkSgGDfCYDStZBWneWCaPSqES09WUd9XQXOR1U/cl6rmYSt/MdLeyEIdCCENi+FsqhHCtHxo9AUarS4wJ3pqistbQuNtIoGh0uBYAUM1JUgU34VEc47ksb+SuT82TtK4b/7HIJumNzH3bsnpsEgs7X/gEm2RlKcNEEFs7C7e+VdEJI7Am66FTx9+xz4abEHjGY2N7m60kPtSuHLFUej2v76upzbWcbSc07NOFi1lResKY20KY8nMtnW7wCTvd+LS2eBdslHQz5GKvzngG78DIpImuVOx693ts9JclmAOztzHCg1ur1U2XODg93uKYtl6gbdih1Q7XeWaLjBrYdpiF00S1tEF9Q0sKv16gOuKpdaaMytSMQRljhAKA9VjuXJRZUW7rTh/NxUJIgCs5zh3S865qE2oMyfYWXz4hp3SHdNUmbazBPvzANPfIgfhRFdBR3L5pFRGDl2nfjZclwCOckjhG/u2DrTGuJIFDC1KTqraoRxXQwIBlUQdPsuKVLFX1PdigjUK2rEA1E/BGvpVcbgGK4ZXoNI9A70BDM9kZHetZnyrpp1Qnp5xF/rZSsn2Jj6wWAW6c+oiiMAF46vOgBEszyEBt82M0vSu7k3vb6oNPh41NwGga7++pXyd8PCpxuBOdY0g/0K3TdH4wzzmiwHScbZ4gzzb+aFQ7aL2brWN76pHAa5illnjr6FRVgaz6ySdqHPhR16VzuVzKcpEqQ/QZ2A5zQL0UfpLWf+bNL11VcHufQkdgOGWfwp6fmmnrcLpR1PHZcmxstlxvmlqfuwpVVlgrb1qrJ+ZylalxW2W68OOxblpWqMN6UzbxPnSud9I5jK8clTNsyK31TME1XDIyBdc+jibyGEjHQ5BxHGTN9tGTQGId7hHBtmKWbB6PEK3IvlAj3uqdf+3pelnIqpZV0PR+KCpQ6eZrT9brZrlod8/g0d5Rr5rgxXSn8Uae8zvpKNbT9+MxjPXd76Z+aoNdNyfhW0xXN1y27x+pv8SBeGtLQxzjUNXEqWE45Fi4Fvm6Vv0MN7btQylWU1HmUk0kn65K/4xNN4c3j+zOlcSz1pW/vUYkDsZtO/AZMV2kSOtSZKNayTmNGzDlQ++zeTu94T1rYe2xfopxwoh6QDw86uh6Xoo1WLVbLHwmSngUQSZXYOLKladX2HEsGR8ue+lsokbmUg0DkjEyN/EqJDLl4joQOKpfoR6YIxJHrbvBHBKKONhq4faJhl7kQW+XrxADi4rGmKMIob7B+kJMJmkdPCmU56QZT/5YqKxO0O+u5OfSf7xgz0+JfLpVdwtHqSe24ZP2PdcP69yb2nq3mcOEjFGzZBwLnVqOU/URxldFEo+6p24bdM5itL1nRyyXqlKvZXC+HR0IVsXh6bLI1pOiLj01u9htBtqgx6hd4YJ0LeHRCXuV6OHCoaOnk59B23JgVITCkEjVCYPpR8bsg2K4c9MQz0dPlYH9ZzHYjBzY/mhc2wtR15mscvHFPmivqvjrOWhyqjwivSoA9zxtWaeBOzyMB7P4W3GfJgzkWUPKRxW96mNgDuMNWe+0nGC8IR9ZWRdrComdEkq1FFr1tCQE5ga5EALdBPfj9y/k7L6tgF+Wq4lUIX7PUUbUKUIb4cE5MKEysul2wpFvnSsZ1SxTH0rtlPBbbe2IuVRIaJ1fo7I0e4ld492ONHEq0mUj8wDAGAZejCiP4TDjozyAF0Ul7y9B4Kpskj3YHWCvhN8Xy3TqacE61jAn9jo4XwN2VCWYW7M8h2mbpktGTts8UWTMlJ9lawOKx8bCKFf9wBhXPYITcsx7q3wLhgOcp3zI/vJWb2kldlPwXfAa/x4tszusFWpLIfAJ466Os1xNl3n5F3XEY5tFAcvHkvhw3+e11WRycI9s2h7KTFTepRC016RZW5U42JVbv8UtzSM8aSmEU+S4p6YOuiF6E5k8iR/VnNpRee4iIB7dCUNLvLWHDo9xHEcsoZSowdg95rsQ18Uu72k9FVVQypsiW/mGXt2A3qUc1ilWiE6Ye+JEuy7K5IQb4QkTzp1AT0Ntd/XpCajtBCJQk26+u++j3MV0Lq+uMlkKLxv1eyT9bjuKbtqLekYAERs1w6ETytK6D8NuH7XB+QBJiOih0ZFnJt2oSgZ9M5/u96CR07WsPdfami+MQ9XXMGYE0QjshT5npdmNkp9FPS+q4OximuYzsVSNDpVxsPI1VJ26PTvwn+mSt5SDnoyfmBBx2r6dYj41P43b06HTK4vbt+i0nV4jUE0Str/vByEuGpfcthXOhcgreZNKz5gW7/X70GaxDvqMOGrfuGfMaNSALjdf/nCIS5VH7mslY09qw1F5ck5TPQeHnxnCxw7UWpssqVnTauIqWKDA232f70WRX6vGJU3LuVZniFKCHP3Sj8Wwm7Ld1c9YOKRt48hPKxvT40ufLkIGuYoknnMGR/Wi7gye13oFhycBnCDfsM068IH73LJMZwcz8tE8K5pJM/VqEUyLRVH5jq80NJO1emy7s9TJO/DYxuOzrk2NWEdmXVtFnyFEaxPU25GO2yx9cuIh0MobhpAiCoA6wX2/2EUtrtWA8W28HZitXk9lueklfbUqp97NfR0N6eHBZ91aUvQOmoc49tcZ7kVMURRySgiY5Nh/Qs6YLF7X4SWGLo4WAItHAljEwigEhZ7GEQ2P2oNc1GW6EX7nUpRqFvSFzOGDFUFZvMrkWuSeer1ji/Bh3zNtV5XE76RiWcdXna6/PBnftyk5IZf9iY376s7PUuH0kFLOcULJkfb+Kq2q4NFcykoGn6Rq8u6+EYiYgTZf5J4NfbHujA12IMWetbWjZjmPj15Sg2HylqKX2hZg3alYQ2oH5SonO0xCFVQkLPIckmWfJRPGIbxwhEByI9avIW16GRki55N0M0Y8vUpnvg3w/7SEjr17pdtEA8NIyb0Pz6VFn3ZH+dtrvxAZXC17l4pLPbbOsW9vxgAaWqT688CjWqE+O66ydOk3+y9sd2XSZrx3a3yHW1oSh/YB2oz3UZ6vKA754e57pOuDXeeQpO6Vrm8vbQHuaN2Nr+zo5dfhE1R2dKvoDxOiQ3bUVusTCVpZ8CRb16VYnsLLP1g35RkIZ2+vaqCTqJoMz5/Xu54cUlXZqMKVTveB2G+QxdZiR9s+L54j15E9/yziPAaDF84eYEyPfz4VoHctZAbCcjC9VpTX61L6Zjka8MrWqijuepY9OW+MruaY0+CQa8t6ozI9tzJ8e+k+LsQTFwniRXwXx0Aegzmn8gYYoX2j4FMpl2kziHI/H3wi88/FIs09x9Bjt1Sz4cxuzAFkEY8TDAcGv4MxWr1pjbHHLJKmhWi8byl6glEkowcDOjR07/q4Pedl2ixONak5Uh3tQVJG5Kib/afztJbBw7ScNd4NUYpZCrbn2UVdyiytvwKcy9VUNpG+lYGnw0M4425ByDvwoxE3x5TD1DMKSjBBBEckRFifecZOaCMQFNMoUpk1qsWyNg/d1NlpzywJZqDYqSQdFDPPDFtsb40P+4mZ6ucFJMdHRsKP5KvgPFstgvPG9XFezJpZu2ru7n5QvaejzdF+oieIkYSjWoy7TS4wQOGe7/TdRR38o6NyNLoRKPI9Os2cuLVM1FN0+3y2mk7F2+j9ZzeJep5tjR6UnG52B/MI5piaOB6IY+7Enaq5pEMHe6eWJNzdjdn1bWDg7/19H7Y++2wX5vwMlLJiuspU5uJX8HxEWrZjo5PhBuJde1tXUExPmXvh1q7NlM+7I54pJYWEJOEJyAZP4se1a3Pr88UMrS72pYQkCiM4MjHgYRJpBCeNPEq5e81zwf6loMap0VKsnzX7mZyUIngqXsHbzJWRuntaLKu0CkRdy3zl21qhy0vanU2w3dHg6+UdVdn+5zx011TR4NlwLokiFILiCbYaQNUpGs51+waSOKIcxRFSLEkGGXELhu8v5xKezeAnarx4Wfo2sf9zqn93ryi3d4GPnLrAa5MfR+Vxayfd2AcExNizGorbE2xpCDol5pv7LZOpB7j7yf8B+bC+ug=='
baseline_ref_path = EXP_ROOT / 'baseline_bao_cao_mau.json'
if baseline_ref_path.is_file():
    BASELINE_JSON = json.loads(baseline_ref_path.read_text(encoding='utf-8'))
else:
    BASELINE_JSON = json.loads(zlib.decompress(base64.b64decode(BASELINE_B64)).decode('utf-8'))
    baseline_ref_path.write_text(json.dumps(BASELINE_JSON, ensure_ascii=False, indent=2), encoding='utf-8')

LABELS_90 = [x['original_label'] for x in BASELINE_JSON['labels_vi']]
assert len(LABELS_90) == 90

"""Shared pure-Python pipeline helpers for from-scratch 90-class experiments."""


import json
import math
import os
import random
import re
import shutil
import tempfile
import time
import zipfile
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd


def is_drive_fuse_path(path: Path) -> bool:
    return '/content/drive' in str(path).replace('\\', '/')


def safe_write_bytes(path: Path, data: bytes, retries: int = 5) -> None:
    """Write file safely: in-place with retries on Colab Drive FUSE, atomic replace locally."""
    last_err: Exception | None = None
    for attempt in range(retries):
        try:
            path.parent.mkdir(parents=True, exist_ok=True)
            if is_drive_fuse_path(path):
                with open(path, 'wb') as f:
                    f.write(data)
                    f.flush()
                    os.fsync(f.fileno())
            else:
                tmp = path.with_name(f'.tmp_{os.getpid()}_{path.name}')
                with open(tmp, 'wb') as f:
                    f.write(data)
                    f.flush()
                os.replace(tmp, path)
            return
        except OSError as exc:
            last_err = exc
            time.sleep(0.4 * (attempt + 1))
    assert last_err is not None
    raise last_err


def atomic_write_text(path: Path, text: str) -> None:
    safe_write_bytes(path, text.encode('utf-8'))


def safe_copy_file(src: Path, dst: Path, retries: int = 5) -> None:
    """Copy bytes from local staged file into destination without Drive FUSE .tmp rename."""
    last_err: Exception | None = None
    for attempt in range(retries):
        try:
            dst.parent.mkdir(parents=True, exist_ok=True)
            if is_drive_fuse_path(dst):
                with open(src, 'rb') as rf, open(dst, 'wb') as wf:
                    shutil.copyfileobj(rf, wf, length=1024 * 1024)
                    wf.flush()
                    os.fsync(wf.fileno())
            else:
                tmp = dst.with_name(f'.tmp_{os.getpid()}_{dst.name}')
                shutil.copy2(src, tmp)
                os.replace(tmp, dst)
            return
        except OSError as exc:
            last_err = exc
            time.sleep(0.5 * (attempt + 1))
    assert last_err is not None
    raise last_err


def safe_save_csv(df: pd.DataFrame, dst: Path, **to_csv_kwargs: Any) -> None:
    with tempfile.NamedTemporaryFile(suffix='.csv', delete=False) as tfp:
        tmp_path = Path(tfp.name)
    try:
        df.to_csv(tmp_path, **to_csv_kwargs)
        safe_copy_file(tmp_path, dst)
    finally:
        tmp_path.unlink(missing_ok=True)


def safe_save_excel(sheets: dict[str, tuple[pd.DataFrame, bool]], dst: Path) -> None:
    with tempfile.NamedTemporaryFile(suffix='.xlsx', delete=False) as tfp:
        tmp_path = Path(tfp.name)
    try:
        with pd.ExcelWriter(tmp_path) as writer:
            for sh_name, (df, keep_index) in sheets.items():
                df.to_excel(writer, sheet_name=sh_name, index=keep_index)
        safe_copy_file(tmp_path, dst)
    finally:
        tmp_path.unlink(missing_ok=True)


def read_phase_epoch_stats(phase_dir: Path, tracker: 'SaturationTracker') -> tuple[int | None, int | None]:
    """Recover best_epoch and stopped_epoch from tracker with history.csv fallback."""
    best_ep = tracker.best_epoch
    stopped_ep = tracker.next_epoch or None
    hist_path = phase_dir / 'history.csv'
    if hist_path.is_file():
        try:
            df_h = pd.read_csv(hist_path)
            if not df_h.empty and 'epoch' in df_h.columns:
                max_ep = int(df_h['epoch'].max())
                if stopped_ep is None or max_ep > stopped_ep:
                    stopped_ep = max_ep
                if 'val_loss' in df_h.columns and df_h['val_loss'].notna().any():
                    best_ep_h = int(df_h.loc[df_h['val_loss'].idxmin(), 'epoch'])
                    if best_ep is None or max_ep > (tracker.next_epoch or 0):
                        best_ep = best_ep_h
        except Exception:
            pass
    return best_ep, stopped_ep


def build_ckpt_zip_name(run_key: str, stage_num: int, phase: str, ep1: int) -> str:
    return f'ckpt_{run_key}_{int(stage_num):02d}_{phase}_ep{int(ep1):03d}.zip'


def pack_dir_to_ckpt_zip(
    local_dir: Path,
    ckpt_dir: Path,
    run_key: str,
    stage_num: int,
    phase: str,
    ep1: int,
) -> Path:
    """Bundle local training state into a unique monotonic zip inside ckpt_dir without FUSE overwrites."""
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    zip_name = build_ckpt_zip_name(run_key, stage_num, phase, ep1)
    dst_zip = ckpt_dir / zip_name
    with tempfile.NamedTemporaryFile(suffix='.zip', delete=False) as tfp:
        tmp_zip = Path(tfp.name)
    try:
        with zipfile.ZipFile(tmp_zip, 'w', compression=zipfile.ZIP_STORED) as zf:
            for item in sorted(local_dir.rglob('*')):
                if item.is_file():
                    rel = item.relative_to(local_dir).as_posix()
                    zf.write(item, arcname=rel)
        safe_copy_file(tmp_zip, dst_zip)
    finally:
        tmp_zip.unlink(missing_ok=True)

    for old in sorted(ckpt_dir.glob(f'ckpt_{run_key}_*.zip')):
        if old.name != zip_name:
            try:
                old.unlink()
            except OSError:
                pass
    return dst_zip


def restore_dir_from_latest_ckpt_zip(local_dir: Path, ckpt_dir: Path, run_key: str) -> Path | None:
    """Restore local_dir from the newest non-corrupt checkpoint zip in ckpt_dir."""
    if not ckpt_dir.exists():
        return None
    candidates = sorted(ckpt_dir.glob(f'ckpt_{run_key}_*.zip'), reverse=True)
    for zp in candidates:
        try:
            with zipfile.ZipFile(zp, 'r') as zf:
                if zf.testzip() is not None:
                    continue
                local_dir.mkdir(parents=True, exist_ok=True)
                zf.extractall(local_dir)
                return zp
        except (OSError, zipfile.BadZipFile):
            continue
    return None


def save_dir_to_flat_zip(src_dir: Path, dst_zip: Path, exclude_prefixes: tuple[str, ...] = ()) -> Path:
    """Pack a completed local run directory into a single flat zip archive on Drive."""
    dst_zip.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile(suffix='.zip', delete=False) as tfp:
        tmp_zip = Path(tfp.name)
    try:
        with zipfile.ZipFile(tmp_zip, 'w', compression=zipfile.ZIP_STORED) as zf:
            for item in sorted(src_dir.rglob('*')):
                if not item.is_file():
                    continue
                rel = item.relative_to(src_dir).as_posix()
                if any(rel.startswith(pfx) for pfx in exclude_prefixes):
                    continue
                zf.write(item, arcname=rel)
        safe_copy_file(tmp_zip, dst_zip)
    finally:
        tmp_zip.unlink(missing_ok=True)
    return dst_zip


def read_eval_from_dir_or_bundle(
    run_dir: Path, bundle_zip: Path
) -> tuple[dict[str, Any] | None, pd.DataFrame | None]:
    """Load metrics.json and classification_report.csv from run_dir or fallback complete zip."""
    ev = run_dir / 'evaluation_report'
    m_file = ev / 'metrics.json'
    r_file = ev / 'classification_report.csv'
    if m_file.is_file() and r_file.is_file():
        return json.loads(m_file.read_text(encoding='utf-8')), pd.read_csv(r_file, index_col=0)
    if bundle_zip.is_file():
        try:
            with zipfile.ZipFile(bundle_zip, 'r') as zf:
                names = set(zf.namelist())
                m_arc = 'evaluation_report/metrics.json'
                r_arc = 'evaluation_report/classification_report.csv'
                if m_arc in names and r_arc in names:
                    metrics = json.loads(zf.read(m_arc).decode('utf-8'))
                    with zf.open(r_arc) as fp:
                        rep = pd.read_csv(fp, index_col=0)
                    return metrics, rep
        except (OSError, zipfile.BadZipFile):
            pass
    return None, None


CLIP_NAME_RE = re.compile(r'^(clip\d+)_', re.IGNORECASE)
EXCEL_METRIC_MAP = [
    ('Top-1 Accuracy', 'top1_accuracy'),
    ('Top-3 Accuracy', 'top3_accuracy'),
    ('Weighted Precision', 'weighted_precision'),
    ('Weighted Recall', 'weighted_recall'),
    ('Weighted F1-Score', 'weighted_f1'),
    ('Macro Precision', 'macro_precision'),
    ('Macro Recall', 'macro_recall'),
    ('Macro F1-Score', 'macro_f1'),
]


def split_real_images_leak_free(
    paths: list[Path], cls_name: str, seed: int
) -> dict[str, list[Path]]:
    """Split supplemented real images into train/val/test without frame or crop leakage."""
    ordered = sorted(set(paths), key=lambda p: p.name)
    n = len(ordered)
    if n == 0:
        return {'train': [], 'val': [], 'test': []}
    if n <= 4:
        return {'train': ordered, 'val': [], 'test': []}

    rng = random.Random(int(seed) + sum(ord(ch) for ch in cls_name) * 31 + 7)
    clip_count = sum(1 for p in ordered if CLIP_NAME_RE.match(p.name))

    # Chronological video frames: partition into ~10 contiguous temporal chunks and select
    # non-adjacent val/test chunks so consecutive frames never cross split boundaries.
    if clip_count >= max(8, int(n * 0.6)):
        n_blocks = min(10, max(5, n // 3))
        blocks: list[list[Path]] = [[] for _ in range(n_blocks)]
        for idx, p in enumerate(ordered):
            blk = min(n_blocks - 1, (idx * n_blocks) // n)
            blocks[blk].append(p)
        blocks = [b for b in blocks if b]
        nb = len(blocks)
        if nb >= 5:
            val_idx = rng.randrange(nb)
            candidates = [i for i in range(nb) if abs(i - val_idx) >= 2]
            if not candidates:
                candidates = [i for i in range(nb) if i != val_idx]
            test_idx = rng.choice(candidates)
            tr, va, te = [], [], []
            for idx, grp in enumerate(blocks):
                if idx == val_idx:
                    va.extend(grp)
                elif idx == test_idx:
                    te.extend(grp)
                else:
                    tr.extend(grp)
            return {'train': tr, 'val': va, 'test': te}

    # Standalone photos: group base photo + *_center crop together before splitting.
    groups_map: dict[str, list[Path]] = {}
    for p in ordered:
        stem = p.stem[:-7] if p.stem.endswith('_center') else p.stem
        groups_map.setdefault(stem, []).append(p)

    groups = [groups_map[k] for k in sorted(groups_map)]
    rng.shuffle(groups)
    if len(groups) < 3:
        return {'train': [p for g in groups for p in g], 'val': [], 'test': []}

    val_grp = groups[0]
    test_grp = groups[1]
    train_grps = groups[2:]
    return {
        'train': [p for g in train_grps for p in g],
        'val': list(val_grp),
        'test': list(test_grp),
    }


def build_dataset_split_plan(
    base_map: dict[str, list[Path]],
    supp_map: dict[str, list[Path]],
    seed: int,
) -> tuple[dict[str, dict[str, list[Path]]], pd.DataFrame]:
    """Create a reproducible stratified 80/10/10 plan combining base + supplemented 90 classes."""
    class_folders = sorted(base_map.keys())
    plan: dict[str, dict[str, list[Path]]] = {}
    rows: list[dict[str, Any]] = []
    rng = random.Random(int(seed))

    for cls_name in class_folders:
        base_imgs = sorted(set(base_map[cls_name]), key=lambda p: p.name)
        rng.shuffle(base_imgs)
        n_base = len(base_imgs)
        n_tr = int(n_base * 0.80)
        n_va = int(n_base * 0.10)
        if n_va == 0 and n_base >= 3:
            n_va = 1

        real_imgs = supp_map.get(cls_name, [])
        real_split = split_real_images_leak_free(real_imgs, cls_name, int(seed))

        cls_plan = {
            'train_base': base_imgs[:n_tr],
            'val_base': base_imgs[n_tr:n_tr + n_va],
            'test_base': base_imgs[n_tr + n_va:],
            'train_real': real_split['train'],
            'val_real': real_split['val'],
            'test_real': real_split['test'],
        }
        plan[cls_name] = cls_plan

        rows.append({
            'label': cls_name,
            'train_base': len(cls_plan['train_base']),
            'train_real': len(cls_plan['train_real']),
            'val_base': len(cls_plan['val_base']),
            'val_real': len(cls_plan['val_real']),
            'test_base': len(cls_plan['test_base']),
            'test_real': len(cls_plan['test_real']),
            'total_unique': n_base + len(real_imgs),
        })

    return plan, pd.DataFrame(rows)


def build_real_train_oversample_entries(
    real_train_paths: list[Path],
    max_total: int = 48,
    max_repeat: int = 8,
) -> list[tuple[Path, str]]:
    """Build bounded oversampling copies of real images strictly for the Train split."""
    ordered = sorted(real_train_paths, key=lambda p: p.name)
    n = len(ordered)
    if n == 0:
        return []
    repeat = max(1, min(max_repeat, max_total // n if max_total >= n else 1))
    entries: list[tuple[Path, str]] = []
    for rep in range(repeat):
        for idx, src in enumerate(ordered):
            if len(entries) >= max_total and rep > 0:
                break
            dest_name = f'real_r{rep:02d}_{idx:03d}_{src.name}'
            entries.append((src, dest_name))
    return entries


def compute_damped_class_weights(counts: np.ndarray) -> dict[int, float]:
    """Compute square-root damped balanced class weights clipped to [0.65, 2.20]."""
    arr = np.asarray(counts, dtype=np.float64)
    mean_c = float(np.mean(arr)) if len(arr) > 0 else 1.0
    damped = np.clip((mean_c / np.maximum(arr, 1.0)) ** 0.5, 0.65, 2.20)
    return {i: float(w) for i, w in enumerate(damped)}


class SaturationTracker:
    """Resumable EarlyStopping + ReduceLROnPlateau state tracker on val_loss."""

    def __init__(
        self,
        phase: str,
        max_epochs: int,
        init_lr: float,
        min_lr: float,
        lr_factor: float = 0.5,
        lr_patience: int = 3,
        es_patience: int = 8,
        min_delta: float = 1e-4,
    ) -> None:
        self.phase = phase
        self.max_epochs = int(max_epochs)
        self.init_lr = float(init_lr)
        self.min_lr = float(min_lr)
        self.lr_factor = float(lr_factor)
        self.lr_patience = int(lr_patience)
        self.es_patience = int(es_patience)
        self.min_delta = float(min_delta)

        self.current_lr = float(init_lr)
        self.next_epoch = 0
        self.best_val_loss: float | None = None
        self.ref_val_loss: float | None = None
        self.best_val_acc: float | None = None
        self.best_epoch: int | None = None
        self.wait_es = 0
        self.wait_lr = 0
        self.lr_reductions = 0
        self.stopped = False
        self.stop_reason: str | None = None

    def step(self, epoch_idx: int, val_loss: float, val_acc: float) -> dict[str, Any]:
        val_loss = float(val_loss)
        val_acc = float(val_acc)
        if not math.isfinite(val_loss):
            self.stopped = True
            self.stop_reason = 'non_finite_loss'
            return {'save_best': False, 'new_lr': None, 'stop': True}

        save_best = False
        if self.best_val_loss is None or val_loss < self.best_val_loss:
            self.best_val_loss = val_loss
            self.best_val_acc = val_acc
            self.best_epoch = int(epoch_idx) + 1
            save_best = True

        new_lr = None
        if self.ref_val_loss is None or (self.ref_val_loss - val_loss) > self.min_delta:
            self.ref_val_loss = val_loss
            self.wait_es = 0
            self.wait_lr = 0
        else:
            self.wait_es += 1
            self.wait_lr += 1
            if self.wait_lr >= self.lr_patience and self.current_lr > self.min_lr + 1e-12:
                next_lr = max(self.min_lr, self.current_lr * self.lr_factor)
                if next_lr < self.current_lr - 1e-12:
                    self.current_lr = next_lr
                    self.lr_reductions += 1
                    new_lr = next_lr
                self.wait_lr = 0

        self.next_epoch = int(epoch_idx) + 1
        if self.wait_es >= self.es_patience:
            self.stopped = True
            self.stop_reason = 'early_stopping_saturated'
        elif self.next_epoch >= self.max_epochs:
            self.stopped = True
            self.stop_reason = 'max_epochs_reached'

        return {
            'save_best': save_best,
            'new_lr': new_lr,
            'stop': self.stopped,
        }

    def to_dict(self) -> dict[str, Any]:
        return {
            'phase': self.phase,
            'max_epochs': self.max_epochs,
            'init_lr': self.init_lr,
            'min_lr': self.min_lr,
            'lr_factor': self.lr_factor,
            'lr_patience': self.lr_patience,
            'es_patience': self.es_patience,
            'min_delta': self.min_delta,
            'current_lr': self.current_lr,
            'next_epoch': self.next_epoch,
            'best_val_loss': self.best_val_loss,
            'ref_val_loss': self.ref_val_loss,
            'best_val_acc': self.best_val_acc,
            'best_epoch': self.best_epoch,
            'wait_es': self.wait_es,
            'wait_lr': self.wait_lr,
            'lr_reductions': self.lr_reductions,
            'stopped': self.stopped,
            'stop_reason': self.stop_reason,
        }

    @classmethod
    def from_dict(cls, data: dict[str, Any]) -> 'SaturationTracker':
        obj = cls(
            phase=data['phase'],
            max_epochs=data['max_epochs'],
            init_lr=data['init_lr'],
            min_lr=data['min_lr'],
            lr_factor=data['lr_factor'],
            lr_patience=data['lr_patience'],
            es_patience=data['es_patience'],
            min_delta=data['min_delta'],
        )
        for attr in (
            'current_lr',
            'next_epoch',
            'best_val_loss',
            'ref_val_loss',
            'best_val_acc',
            'best_epoch',
            'wait_es',
            'wait_lr',
            'lr_reductions',
            'stopped',
            'stop_reason',
        ):
            setattr(obj, attr, data.get(attr))
        return obj


def _fmt_pct_mean_std(vals: list[float]) -> tuple[float, float, str, str]:
    arr = np.asarray(vals, dtype=float) * 100.0
    m = float(np.mean(arr))
    s = float(np.std(arr, ddof=1)) if len(arr) > 1 else 0.0
    return m, s, f'{m:.2f} ± {s:.2f}%', f'${m:.2f} \\pm {s:.2f}\\%$'


def build_3seed_comparison_table(
    seed_metrics: dict[int, dict[str, Any]],
    baseline_json: dict[str, Any],
    seeds: list[int] | None = None,
) -> pd.DataFrame:
    """Compare new from-scratch 3-seed metrics (99, 123, 999) against reference baseline reports."""
    seeds = list(seeds or [99, 123, 999])
    available = [s for s in seeds if s in seed_metrics]
    if not available:
        return pd.DataFrame()

    ref_seeds = baseline_json.get('seeds_baseline', {})
    ref_42_123_999 = baseline_json.get('reported_seed_groups', {}).get('42_123_999', {})

    rows = []
    for label, key in EXCEL_METRIC_MAP:
        vals_new = [float(seed_metrics[s][key]) for s in available]
        m_new, _, str_new, latex_new = _fmt_pct_mean_std(vals_new)

        vals_ref_same = [
            float(ref_seeds[str(s)][key]) for s in available if str(s) in ref_seeds and key in ref_seeds[str(s)]
        ]
        if vals_ref_same:
            m_ref_same, _, str_ref_same, _ = _fmt_pct_mean_std(vals_ref_same)
            delta_same = m_new - m_ref_same
        else:
            str_ref_same = '—'
            delta_same = None

        ref_excel = ref_42_123_999.get(label, {})
        str_excel_42_123_999 = ref_excel.get('mean_std_str', '—')

        row: dict[str, Any] = {'Chỉ số đánh giá': label}
        for s in seeds:
            if s in seed_metrics:
                row[f'Seed {s} Mới (%)'] = round(float(seed_metrics[s][key]) * 100.0, 2)
            else:
                row[f'Seed {s} Mới (%)'] = None
            if str(s) in ref_seeds and key in ref_seeds[str(s)]:
                row[f'Seed {s} Mẫu (%)'] = round(float(ref_seeds[str(s)][key]) * 100.0, 2)
            else:
                row[f'Seed {s} Mẫu (%)'] = None
        row['Mean ± Std (Từ số 0)'] = str_new
        row['Mean ± Std (Mẫu cùng Seed)'] = str_ref_same
        row['Mean ± Std (Báo cáo mẫu 42/123/999)'] = str_excel_42_123_999
        row['Chênh lệch Top-1/Metric (Mới - Mẫu)'] = round(delta_same, 2) if delta_same is not None else None
        row['Mã LaTeX (Overleaf)'] = latex_new
        rows.append(row)

    return pd.DataFrame(rows)


def build_90_species_comparison_table(
    seed_reports: dict[int, pd.DataFrame],
    baseline_json: dict[str, Any],
    seeds: list[int] | None = None,
) -> pd.DataFrame:
    """Build full 90-species side-by-side table comparing new from-scratch runs vs baseline report."""
    seeds = list(seeds or [99, 123, 999])
    available = [s for s in seeds if s in seed_reports]
    labels_vi = {x['original_label']: x['vietnamese_name'] for x in baseline_json['labels_vi']}
    supp_set = set(baseline_json.get('supplemented_8_classes', []))
    per_cls_base = baseline_json.get('per_class_baseline', {})

    rows = []
    for idx, item in enumerate(baseline_json['labels_vi'], 1):
        lbl = item['original_label']
        vi = labels_vi.get(lbl, lbl)
        is_supp = lbl in supp_set
        base_cls = per_cls_base.get(lbl, {})

        new_f1s = []
        new_ps = []
        new_rs = []
        new_sups = []
        row: dict[str, Any] = {
            'STT': idx,
            'Tên tiếng Anh / Khoa học': lbl,
            'Tên tiếng Việt': vi,
            'Nhóm bổ sung': 'Bổ sung thực tế (8 loài)' if is_supp else 'Lớp nền (90 loài)',
        }
        for s in seeds:
            if s in seed_reports and lbl in seed_reports[s].index:
                r = seed_reports[s].loc[lbl]
                p_val = float(r['precision'])
                rec_val = float(r['recall'])
                f1_val = float(r['f1-score'])
                sup_val = int(round(float(r['support'])))
                row[f'P Mới ({s})'] = round(p_val * 100.0, 2)
                row[f'R Mới ({s})'] = round(rec_val * 100.0, 2)
                row[f'F1 Mới ({s})'] = round(f1_val * 100.0, 2)
                row[f'Test ({s})'] = sup_val
                new_ps.append(p_val)
                new_rs.append(rec_val)
                new_f1s.append(f1_val)
                new_sups.append(sup_val)
            else:
                row[f'P Mới ({s})'] = None
                row[f'R Mới ({s})'] = None
                row[f'F1 Mới ({s})'] = None
                row[f'Test ({s})'] = None

        f1_ref_mean = float(base_cls.get('mean_f1', 0.0)) * 100.0
        row['F1 TB Mẫu (99/123/999) %'] = round(f1_ref_mean, 2)
        if new_f1s:
            m_f1 = float(np.mean(new_f1s) * 100.0)
            sd_f1 = float(np.std(new_f1s, ddof=1) * 100.0) if len(new_f1s) > 1 else 0.0
            row['F1 TB Mới (%)'] = round(m_f1, 2)
            row['Std F1 Mới (%)'] = round(sd_f1, 2)
            row['Chênh lệch F1 (Mới - Mẫu)'] = round(m_f1 - f1_ref_mean, 2)
        else:
            row['F1 TB Mới (%)'] = None
            row['Std F1 Mới (%)'] = None
            row['Chênh lệch F1 (Mới - Mẫu)'] = None
        rows.append(row)

    return pd.DataFrame(rows)


def build_4model_comparison_table(
    model_metrics: dict[str, dict[str, Any]],
    baseline_json: dict[str, Any],
) -> pd.DataFrame:
    """Compare 4 freshly trained saturation models against baseline benchmark report."""
    ref_models = baseline_json.get('benchmark_4_models_baseline', {})
    order = ['custom_cnn', 'mobilenet_v2', 'resnet50', 'efficientnet_b0']
    rows = []
    for mk in order:
        if mk not in model_metrics:
            continue
        cur = model_metrics[mk]
        ref = ref_models.get(mk, {})
        top1_cur = float(cur['top1_acc']) * 100.0
        top3_cur = float(cur['top3_acc']) * 100.0
        f1_cur = float(cur['f1']) * 100.0
        top1_ref = float(ref.get('top1_acc', 0.0)) * 100.0 if 'top1_acc' in ref else None
        f1_ref = float(ref.get('f1', 0.0)) * 100.0 if 'f1' in ref else None

        rows.append({
            'model_key': mk,
            'Mô hình': cur.get('model_title', mk),
            'Tham số (M)': cur.get('params_m'),
            'File .keras (MB)': cur.get('size_mb'),
            'Latency T4 (ms)': cur.get('latency_ms'),
            'Epoch tốt nhất / Dừng': f"{cur.get('best_epoch', '—')}/{cur.get('stopped_epoch', '—')}",
            'Top-1 Mới (%)': round(top1_cur, 2),
            'Top-3 Mới (%)': round(top3_cur, 2),
            'Precision Mới (%)': round(float(cur['precision']) * 100.0, 2),
            'Recall Mới (%)': round(float(cur['recall']) * 100.0, 2),
            'F1 Mới (%)': round(f1_cur, 2),
            'Top-1 Mẫu (%)': round(top1_ref, 2) if top1_ref is not None else None,
            'Chênh lệch Top-1 vs Mẫu': round(top1_cur - top1_ref, 2) if top1_ref is not None else None,
            'Chênh lệch F1 vs Mẫu': round(f1_cur - f1_ref, 2) if f1_ref is not None else None,
        })
    return pd.DataFrame(rows)


# ==============================================================================
# === BỘ TRỢ GIÚP CHUNG: NẠP MODEL AN TOÀN, CONTROLLER BÃO HÒA & CHIA DỮ LIỆU ===
# ==============================================================================
EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def find_first_existing(candidates, rglob_pattern=None):
    for p in candidates:
        if p.is_file():
            return p
    if rglob_pattern:
        found = sorted(PROJECT.rglob(rglob_pattern))
        if found:
            return found[0]
    return None

def sanitize_keras_archive(model_path: Path) -> Path:
    """Loại bỏ quantization_config dư thừa trong config.json nếu chuyển phiên bản Keras."""
    clean_path = Path('/tmp') / f'_compat_{os.getpid()}_{model_path.name}'
    changed = False
    with zipfile.ZipFile(model_path, 'r') as zin:
        with zipfile.ZipFile(clean_path, 'w', compression=zipfile.ZIP_DEFLATED) as zout:
            for item in zin.infolist():
                data = zin.read(item.filename)
                if item.filename == 'config.json':
                    cfg = json.loads(data.decode('utf-8'))
                    def _strip(obj):
                        nonlocal changed
                        if isinstance(obj, dict):
                            if 'quantization_config' in obj:
                                obj.pop('quantization_config')
                                changed = True
                            for v in obj.values():
                                _strip(v)
                        elif isinstance(obj, list):
                            for v in obj:
                                _strip(v)
                    _strip(cfg)
                    data = json.dumps(cfg).encode('utf-8')
                zout.writestr(item, data)
    return clean_path if changed else model_path

def compat_load_model(model_path: Path, compile: bool = False):
    clean_p = sanitize_keras_archive(model_path)
    try:
        try:
            return keras.models.load_model(str(clean_p), compile=compile, safe_mode=False)
        except TypeError:
            return keras.models.load_model(str(clean_p), compile=compile)
    finally:
        if clean_p != model_path and clean_p.exists():
            clean_p.unlink(missing_ok=True)

def safe_save_keras(model: keras.Model, target_path: Path, also_copy_to: Path | None = None) -> None:
    """Lưu .keras ra /tmp SSD trước rồi ghi trực tiếp vào Drive (tránh lỗi .tmp/trùng file của Drive FUSE)."""
    with tempfile.NamedTemporaryFile(suffix='.keras', delete=False) as tfp:
        local_tmp = Path(tfp.name)
    try:
        model.save(str(local_tmp), include_optimizer=False)
        safe_copy_file(local_tmp, target_path)
        if also_copy_to is not None:
            safe_copy_file(local_tmp, also_copy_to)
    finally:
        local_tmp.unlink(missing_ok=True)

def assign_optimizer_lr(optimizer, new_lr: float) -> None:
    lr_obj = getattr(optimizer, 'learning_rate', None)
    if lr_obj is None:
        return
    if hasattr(lr_obj, 'assign'):
        lr_obj.assign(float(new_lr))
    else:
        optimizer.learning_rate = float(new_lr)

class ResumableSaturationCallback(keras.callbacks.Callback):
    """Lưu checkpoint mỗi epoch vào SSD /content cục bộ + đồng bộ file .zip đơn nhất lên Drive."""
    def __init__(
        self,
        phase_dir: Path,
        tracker: SaturationTracker,
        local_run_dir: Path | None = None,
        ckpt_archive_dir: Path | None = None,
        run_key: str | None = None,
        stage_num: int = 1,
        sync_every: int = 3,
    ):
        super().__init__()
        self.phase_dir = Path(phase_dir)
        self.phase_dir.mkdir(parents=True, exist_ok=True)
        self.tracker = tracker
        self.local_run_dir = Path(local_run_dir) if local_run_dir is not None else self.phase_dir.parent
        self.ckpt_archive_dir = Path(ckpt_archive_dir) if ckpt_archive_dir is not None else None
        self.run_key = run_key
        self.stage_num = int(stage_num)
        self.sync_every = max(1, int(sync_every))
        self.state_path = self.phase_dir / 'progress.json'
        self.last_path = self.phase_dir / 'last_epoch.keras'
        self.best_path = self.phase_dir / 'best_model.keras'
        self.hist_path = self.phase_dir / 'history.csv'
        self.history_cols = [
            'epoch', 'loss', 'accuracy', 'top3_accuracy',
            'val_loss', 'val_accuracy', 'val_top3_accuracy', 'lr'
        ]

    def _append_history(self, epoch_idx: int, logs: dict, used_lr: float) -> None:
        ep1 = int(epoch_idx) + 1
        row = {
            'epoch': ep1,
            'loss': float(logs.get('loss', 0.0)),
            'accuracy': float(logs.get('accuracy', 0.0)),
            'top3_accuracy': float(logs.get('top3_accuracy', 0.0)),
            'val_loss': float(logs.get('val_loss', 0.0)),
            'val_accuracy': float(logs.get('val_accuracy', 0.0)),
            'val_top3_accuracy': float(logs.get('val_top3_accuracy', 0.0)),
            'lr': float(used_lr),
        }
        if self.hist_path.exists():
            df_old = pd.read_csv(self.hist_path)
            df_old = df_old[df_old['epoch'] != ep1]
            df_new = pd.concat([df_old, pd.DataFrame([row])], ignore_index=True)
        else:
            df_new = pd.DataFrame([row], columns=self.history_cols)
        safe_save_csv(df_new, self.hist_path, index=False, encoding='utf-8-sig')

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        used_lr = float(self.tracker.current_lr)
        action = self.tracker.step(epoch, logs.get('val_loss', float('nan')), logs.get('val_accuracy', 0.0))
        self._append_history(epoch, logs, used_lr)

        copy_best = self.best_path if (action['save_best'] or not self.best_path.exists()) else None
        safe_save_keras(self.model, self.last_path, also_copy_to=copy_best)

        if action['new_lr'] is not None:
            assign_optimizer_lr(self.model.optimizer, action['new_lr'])
            print(f'\n  📉 Plateau tại Epoch {epoch+1}: giảm LR -> {action["new_lr"]:.2e}')

        atomic_write_text(self.state_path, json.dumps(self.tracker.to_dict(), indent=2))
        ep1 = int(epoch) + 1
        sync_note = ''
        if self.ckpt_archive_dir is not None and self.run_key and (
            action['save_best'] or action['stop'] or (ep1 % self.sync_every == 0)
        ):
            zp = pack_dir_to_ckpt_zip(
                self.local_run_dir, self.ckpt_archive_dir, self.run_key,
                self.stage_num, self.tracker.phase, ep1,
            )
            sync_note = f' | Drive sync: {zp.name}'

        best_s = f'{self.tracker.best_val_loss:.4f}' if self.tracker.best_val_loss is not None else 'N/A'
        print(
            f'  💾 Lưu Epoch {ep1}/{self.tracker.max_epochs} | '
            f'val_loss={float(logs.get("val_loss", 0)):.4f} (best={best_s} @ Ep {self.tracker.best_epoch}) | '
            f'val_acc={float(logs.get("val_accuracy", 0))*100:.2f}% | '
            f'wait_es={self.tracker.wait_es}/{self.tracker.es_patience}{sync_note}'
        )
        if action['stop']:
            print(f'  ⏹️ Dừng huấn luyện ({self.tracker.stop_reason}) ở Epoch {ep1}!')
            self.model.stop_training = True

def locate_subdir(root_dir: Path, target_name: str) -> Path:
    direct = root_dir / target_name
    if direct.is_dir():
        return direct
    matches = [p for p in root_dir.rglob(target_name) if p.is_dir()]
    assert matches, f'Không tìm thấy thư mục {target_name} bên trong {root_dir}!'
    return matches[0]

def scan_base_90_images(raw_base_dir: Path, expected_labels: list[str]) -> dict[str, list[Path]]:
    found = {}
    for root, _, files in os.walk(str(raw_base_dir)):
        folder_name = os.path.basename(root)
        if folder_name.startswith('.'):
            continue
        imgs = sorted(
            [Path(root) / f for f in files if Path(f).suffix.lower() in EXTENSIONS],
            key=lambda p: p.name,
        )
        if imgs and Path(root) != raw_base_dir:
            found.setdefault(folder_name, []).extend(imgs)
    missing = [lbl for lbl in expected_labels if lbl not in found]
    assert not missing, f'Thiếu {len(missing)} lớp trong bộ gốc 90 loài: {missing[:5]}'
    return {lbl: sorted(set(found[lbl]), key=lambda p: p.name) for lbl in expected_labels}

def scan_supplemented_90_images(known_supp_dir: Path, expected_labels: list[str]) -> dict[str, list[Path]]:
    valid_set = set(expected_labels)
    supp = {}
    for d in sorted(known_supp_dir.iterdir()):
        if d.is_dir() and d.name in valid_set:
            imgs = sorted([p for p in d.iterdir() if p.is_file() and p.suffix.lower() in EXTENSIONS], key=lambda p: p.name)
            if imgs:
                supp[d.name] = imgs
    return supp

def _link_or_copy(src: Path, dst: Path) -> None:
    if dst.exists():
        return
    try:
        os.link(src, dst)
    except OSError:
        shutil.copy2(src, dst)

def materialize_split_to_ssd(
    base_map: dict[str, list[Path]],
    supp_map: dict[str, list[Path]],
    seed: int,
    split_dir: Path,
    report_dir: Path,
) -> tuple[pd.DataFrame, dict[str, list[str]]]:
    """Tạo thư mục train/val/test trên /content SSD theo chiến lược không rò rỉ dữ liệu."""
    plan, split_df = build_dataset_split_plan(base_map, supp_map, seed=seed)
    real_test_relpaths = {}
    done_marker = split_dir / f'_split_ready_seed_{seed}.json'

    if not done_marker.exists():
        if split_dir.exists():
            shutil.rmtree(split_dir)
        for cls_name, cls_plan in plan.items():
            tr_dir = split_dir / 'train' / cls_name
            va_dir = split_dir / 'val' / cls_name
            te_dir = split_dir / 'test' / cls_name
            tr_dir.mkdir(parents=True, exist_ok=True)
            va_dir.mkdir(parents=True, exist_ok=True)
            te_dir.mkdir(parents=True, exist_ok=True)

            for p in cls_plan['train_base']:
                _link_or_copy(p, tr_dir / p.name)
            for p, dup_name in build_real_train_oversample_entries(cls_plan['train_real'], max_total=48, max_repeat=8):
                _link_or_copy(p, tr_dir / dup_name)

            for p in cls_plan['val_base']:
                _link_or_copy(p, va_dir / p.name)
            for p in cls_plan['val_real']:
                _link_or_copy(p, va_dir / f'real_{p.name}')

            for p in cls_plan['test_base']:
                _link_or_copy(p, te_dir / p.name)
            cls_real_tests = []
            for p in cls_plan['test_real']:
                dst_name = f'real_{p.name}'
                _link_or_copy(p, te_dir / dst_name)
                cls_real_tests.append(dst_name)
            if cls_real_tests:
                real_test_relpaths[cls_name] = cls_real_tests

        atomic_write_text(done_marker, json.dumps({'seed': seed, 'real_test_relpaths': real_test_relpaths}, ensure_ascii=False, indent=2))
    else:
        meta = json.loads(done_marker.read_text(encoding='utf-8'))
        real_test_relpaths = meta.get('real_test_relpaths', {})

    report_dir.mkdir(parents=True, exist_ok=True)
    safe_save_csv(split_df, report_dir / f'split_counts_tu_so_0_seed_{seed}.csv', index=False, encoding='utf-8-sig')
    return split_df, real_test_relpaths

def make_datasets_bilinear(split_dir: Path, seed: int, batch_size: int = 32, image_size: tuple[int, int] = (224, 224)):
    """Khởi tạo train/val/test chuẩn Bilinear + crop_to_aspect_ratio + Damped Sample Weights."""
    def _load(sub, shuffle):
        return keras.utils.image_dataset_from_directory(
            str(split_dir / sub),
            image_size=image_size,
            interpolation='bilinear',
            crop_to_aspect_ratio=True,
            batch_size=batch_size,
            label_mode='categorical',
            shuffle=shuffle,
            seed=seed,
        )
    train_raw = _load('train', True)
    val_raw = _load('val', False)
    test_raw = _load('test', False)
    class_names = list(train_raw.class_names)
    assert len(class_names) == 90, f'Expected 90 classes, got {len(class_names)}'

    # Tính nhanh số lượng ảnh train từng lớp ngay từ file trên SSD (không cần decode 20.000 ảnh 2 lần)
    counts = np.array([
        sum(1 for f in (split_dir / 'train' / c).iterdir() if f.is_file() and f.suffix.lower() in EXTENSIONS)
        for c in class_names
    ], dtype=np.float32)
    cw_dict = compute_damped_class_weights(counts)
    w_vec = tf.constant([cw_dict[i] for i in range(len(class_names))], dtype=tf.float32)

    def _attach_weight(x, y):
        sw = tf.reduce_sum(tf.cast(y, tf.float32) * w_vec, axis=-1)
        return x, y, sw

    train_ds = train_raw.map(_attach_weight, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)
    val_ds = val_raw.prefetch(tf.data.AUTOTUNE)
    test_ds = test_raw.prefetch(tf.data.AUTOTUNE)
    return train_ds, val_ds, test_ds, test_raw, class_names, counts

def find_backbone_layer(model: keras.Model, backbone_key: str):
    key = backbone_key.lower().replace('_', '')
    for l in model.layers:
        lname = l.name.lower().replace('_', '')
        if isinstance(l, keras.Model) and (key in lname or any(k in lname for k in ('mobilenet', 'efficientnet', 'resnet'))):
            return l
    raise RuntimeError(f'Không tìm thấy tầng backbone {backbone_key} trong model!')

def set_backbone_finetune_trainable(model: keras.Model, backbone_key: str, unfreeze_last: int = 50) -> tuple[int, int]:
    base = find_backbone_layer(model, backbone_key)
    base.trainable = True
    total = len(base.layers)
    cutoff = max(0, total - int(unfreeze_last))
    for l in base.layers[:cutoff]:
        l.trainable = False
    n_unfrozen = 0
    for l in base.layers[cutoff:]:
        if isinstance(l, layers.BatchNormalization):
            l.trainable = False
        else:
            l.trainable = True
            n_unfrozen += 1
    return n_unfrozen, total

def evaluate_90_classes_complete(
    model: keras.Model,
    test_ds,
    test_raw,
    class_names: list[str],
    real_test_relpaths: dict[str, list[str]],
    eval_dir: Path,
    extra_meta: dict,
) -> dict:
    """Đánh giá đầy đủ trên tập Test + tập ảnh thực tế tách riêng + xuất report 90 loài."""
    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

    eval_dir.mkdir(parents=True, exist_ok=True)
    y_true_list, scores_list = [], []
    for imgs, lbls in test_ds:
        probs = model(imgs, training=False).numpy()
        y_true_list.append(np.argmax(lbls.numpy(), axis=1))
        scores_list.append(probs)

    y_true = np.concatenate(y_true_list)
    scores = np.concatenate(scores_list)
    y_pred = np.argmax(scores, axis=1)

    # Đánh giá thêm subset ảnh thực tế chưa thấy (real_test) nếu có
    real_correct, real_total = 0, 0
    cls_to_idx = {c: i for i, c in enumerate(class_names)}
    if hasattr(test_raw, 'file_paths') and len(test_raw.file_paths) == len(y_true):
        for i, fp in enumerate(test_raw.file_paths):
            p = Path(fp)
            cname = p.parent.name
            if p.name in set(real_test_relpaths.get(cname, [])):
                real_total += 1
                if int(y_pred[i]) == cls_to_idx[cname]:
                    real_correct += 1

    metrics_dict = {
        **extra_meta,
        'test_images': int(len(y_true)),
        'correct_predictions': int(np.sum(y_true == y_pred)),
        'incorrect_predictions': int(np.sum(y_true != y_pred)),
        'top1_accuracy': float(accuracy_score(y_true, y_pred)),
        'top3_accuracy': float(np.mean(np.any(np.argsort(scores, axis=1)[:, -3:] == y_true[:, None], axis=1))),
        'macro_precision': float(precision_score(y_true, y_pred, average='macro', zero_division=0)),
        'macro_recall': float(recall_score(y_true, y_pred, average='macro', zero_division=0)),
        'macro_f1': float(f1_score(y_true, y_pred, average='macro', zero_division=0)),
        'weighted_precision': float(precision_score(y_true, y_pred, average='weighted', zero_division=0)),
        'weighted_recall': float(recall_score(y_true, y_pred, average='weighted', zero_division=0)),
        'weighted_f1': float(f1_score(y_true, y_pred, average='weighted', zero_division=0)),
        'real_test_total': int(real_total),
        'real_test_correct': int(real_correct),
        'real_test_acc': float(real_correct / real_total) if real_total > 0 else None,
    }
    atomic_write_text(eval_dir / 'metrics.json', json.dumps(metrics_dict, ensure_ascii=False, indent=2))

    cm = confusion_matrix(y_true, y_pred, labels=np.arange(len(class_names)))
    safe_save_csv(pd.DataFrame(cm, index=class_names, columns=class_names), eval_dir / 'confusion_matrix.csv', encoding='utf-8-sig')

    rep_dict = classification_report(y_true, y_pred, target_names=class_names, output_dict=True, zero_division=0)
    safe_save_csv(pd.DataFrame(rep_dict).transpose(), eval_dir / 'classification_report.csv', encoding='utf-8-sig')
    return metrics_dict

ZIP_30K_PATH = find_first_existing([
    PROJECT / '01_Du_Lieu_Goc' / 'merged_90_30k_chuan.zip',
    PROJECT / '01_Du_Lieu_30k_Va_Labels' / 'merged_v1' / 'merged_90_30k_chuan.zip',
    PROJECT / 'merged_v1' / 'merged_90_30k_chuan.zip',
], rglob_pattern='*merged_90_30k*.zip')
assert ZIP_30K_PATH is not None, 'Không tìm thấy merged_90_30k_chuan.zip trên Google Drive!'

ZIP_BOSUNG_PATH = find_first_existing([
    PROJECT / 'bo_sung_thuc_te.zip',
    PROJECT / '01_Du_Lieu_Goc' / 'bo_sung_thuc_te.zip',
], rglob_pattern='*bo_sung_thuc_te*.zip')
assert ZIP_BOSUNG_PATH is not None, 'Không tìm thấy bo_sung_thuc_te.zip trên Google Drive!'

print('='*74)
print('🚀 ĐÃ THIẾT LẬP PIPELINE TỪ SỐ 0 — 3 SEEDS TỰ ĐỘNG (99, 123, 999)')
print(f'  • TensorFlow    : {tf.__version__} | GPU: {tf.config.list_physical_devices("GPU")}')
print(f'  • Dữ liệu gốc   : {ZIP_30K_PATH}')
print(f'  • Ảnh bổ sung   : {ZIP_BOSUNG_PATH}')
print(f'  • Lưu kết quả ở : {SEEDS_OUT_ROOT}')
print('='*74)


In [ ]:
# ==============================================================================
# === BƯỚC 2: GIẢI NÉN 1 LẦN VÀO SSD /content & KIỂM ĐẾM 90 LOÀI + BỔ SUNG  ===
# ==============================================================================
RAW_BASE_DIR = Path('/content/raw_90_base_30k')
RAW_SUPP_DIR = Path('/content/raw_bo_sung_thuc_te')

if not RAW_BASE_DIR.exists() or len(list(RAW_BASE_DIR.iterdir())) == 0:
    RAW_BASE_DIR.mkdir(parents=True, exist_ok=True)
    print(f'📦 Đang giải nén {ZIP_30K_PATH.name} vào SSD /content (1-2 phút)...')
    !unzip -q "{ZIP_30K_PATH}" -d /content/raw_90_base_30k

if not RAW_SUPP_DIR.exists() or len(list(RAW_SUPP_DIR.iterdir())) == 0:
    RAW_SUPP_DIR.mkdir(parents=True, exist_ok=True)
    print(f'📦 Đang giải nén {ZIP_BOSUNG_PATH.name}...')
    !unzip -q "{ZIP_BOSUNG_PATH}" -d /content/raw_bo_sung_thuc_te

BASE_MAP = scan_base_90_images(RAW_BASE_DIR, LABELS_90)
KNOWN_SUPP_DIR = locate_subdir(RAW_SUPP_DIR, 'da_co_trong_90_loai')
SUPP_MAP = scan_supplemented_90_images(KNOWN_SUPP_DIR, LABELS_90)

tot_base = sum(len(v) for v in BASE_MAP.values())
tot_supp = sum(len(v) for v in SUPP_MAP.values())
print(f'✅ Dữ liệu sạch sẵn sàng: {len(BASE_MAP)} loài gốc ({tot_base:,} ảnh) + {len(SUPP_MAP)} loài bổ sung ({tot_supp:,} ảnh) = {tot_base + tot_supp:,} ảnh.')
for k, v in SUPP_MAP.items():
    print(f'   • {k}: +{len(v)} ảnh thực tế')


In [ ]:
# ==============================================================================
# === BƯỚC 3: ĐỊNH NGHĨA HÀM BUILD & TRAIN MOBILENETV2 CHO TỪNG SEED         ===
# ==============================================================================
def build_fresh_mobilenetv2(seed: int, num_classes: int = 90) -> tuple[keras.Model, keras.Model]:
    inputs = keras.Input(shape=(224, 224, 3), name='image_input')
    x = layers.RandomFlip('horizontal_and_vertical', seed=seed)(inputs)
    x = layers.RandomRotation(0.12, seed=seed)(x)
    x = layers.RandomZoom(0.15, seed=seed)(x)
    x = layers.RandomContrast(0.12, seed=seed)(x)
    x = layers.Rescaling(1.0 / 127.5, offset=-1.0, name='rescaling')(x)
    base = keras.applications.MobileNetV2(input_shape=(224, 224, 3), include_top=False, weights='imagenet')
    base.trainable = False
    feat = base(x, training=False)
    gap = layers.GlobalAveragePooling2D(name='gap')(feat)
    drop = layers.Dropout(0.30, seed=seed, name='dropout')(gap)
    out = layers.Dense(num_classes, activation='softmax', name='pred_90')(drop)
    return keras.Model(inputs, out, name='MobileNetV2_TuSo0_90'), base

def run_single_seed(seed: int) -> dict:
    seed_dir = SEEDS_OUT_ROOT / f'seed_{seed}'
    bundle_zip = COMPLETED_BUNDLES_DIR / f'seed_{seed}_complete.zip'
    run_key = f'seed_{seed}'
    existing_m, _ = read_eval_from_dir_or_bundle(seed_dir, bundle_zip)
    if existing_m is not None and ((seed_dir / 'done.json').is_file() or bundle_zip.is_file()):
        print(f'✅ [Seed {seed}] Đã hoàn tất trước đó, nạp kết quả từ Drive...')
        return existing_m

    print('\n' + '='*78)
    print(f'🚀 BẮT ĐẦU SEED {seed} (TỪ SỐ 0 -> TRAIN TỚI KHI BÃO HÒA)')
    print('='*78)
    random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed); keras.utils.set_random_seed(seed)
    keras.backend.clear_session()

    local_run = Path(f'/content/local_runs/seed_{seed}')
    local_run.mkdir(parents=True, exist_ok=True)
    restored_zip = restore_dir_from_latest_ckpt_zip(local_run, CKPT_ARCHIVE_DIR, run_key)
    if restored_zip is not None:
        print(f'♻️ [Seed {seed}] Đã khôi phục checkpoint từ {restored_zip.name}!')

    split_dir = Path(f'/content/split_tu_so_0_seed_{seed}')
    split_df, real_test_relpaths = materialize_split_to_ssd(BASE_MAP, SUPP_MAP, seed, split_dir, local_run)
    train_ds, val_ds, test_ds, test_raw, class_names, _ = make_datasets_bilinear(split_dir, seed, BATCH_SIZE, IMAGE_SIZE)
    atomic_write_text(local_run / 'class_names.json', json.dumps(class_names, ensure_ascii=False, indent=2))

    loss_fn = keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING)
    metrics_list = [
        keras.metrics.CategoricalAccuracy(name='accuracy'),
        keras.metrics.TopKCategoricalAccuracy(k=3, name='top3_accuracy'),
    ]

    head_dir = local_run / 'head'
    fine_dir = local_run / 'finetune'
    local_eval_dir = local_run / 'evaluation_report'
    local_best_model = local_run / f'best_mobilenetv2_tu_so_0_seed_{seed}.keras'
    drive_best_model = seed_dir / f'best_mobilenetv2_tu_so_0_seed_{seed}.keras'
    head_state_path = head_dir / 'progress.json'
    fine_state_path = fine_dir / 'progress.json'
    tr_head = SaturationTracker.from_dict(json.loads(head_state_path.read_text(encoding='utf-8'))) if head_state_path.is_file() else SaturationTracker(
        'head', MAX_EPOCHS_HEAD, LR_HEAD_INIT, 1e-5, lr_factor=0.5, lr_patience=3, es_patience=7, min_delta=1e-4
    )
    tr_fine = SaturationTracker.from_dict(json.loads(fine_state_path.read_text(encoding='utf-8'))) if fine_state_path.is_file() else SaturationTracker(
        'finetune', MAX_EPOCHS_FINE, LR_FINE_INIT, 1e-6, lr_factor=0.5, lr_patience=3, es_patience=8, min_delta=1e-4
    )

    t0 = time.time()
    if not local_best_model.is_file() and drive_best_model.is_file():
        shutil.copy2(drive_best_model, local_best_model)

    if local_best_model.is_file():
        print(f'♻️ [Seed {seed}] Phát hiện {local_best_model.name} đã train xong — chạy thẳng đánh giá!')
        best_model = compat_load_model(local_best_model, compile=False)
    else:
        # --- GIAI ĐOẠN 1: HEAD WARM-UP ---
        head_dir.mkdir(parents=True, exist_ok=True)
        if not (tr_head.stopped and (head_dir / 'best_model.keras').is_file()):
            if tr_head.next_epoch > 0 and (head_dir / 'last_epoch.keras').is_file():
                print(f'🔄 Nối tiếp Pha Head từ Epoch {tr_head.next_epoch+1} (LR={tr_head.current_lr:.2e})...')
                model = compat_load_model(head_dir / 'last_epoch.keras', compile=False)
                find_backbone_layer(model, 'MobileNetV2').trainable = False
            else:
                model, _ = build_fresh_mobilenetv2(seed, len(class_names))
            model.compile(optimizer=keras.optimizers.Adam(learning_rate=tr_head.current_lr, clipnorm=1.0), loss=loss_fn, metrics=metrics_list)
            cb_h = ResumableSaturationCallback(head_dir, tr_head, local_run, CKPT_ARCHIVE_DIR, run_key, stage_num=1, sync_every=3)
            model.fit(train_ds, validation_data=val_ds, initial_epoch=tr_head.next_epoch, epochs=MAX_EPOCHS_HEAD, callbacks=[cb_h])
        else:
            print(f'✅ Pha Head đã bão hòa ở Epoch {tr_head.best_epoch}/{tr_head.next_epoch}.')

        # --- GIAI ĐOẠN 2: FINE-TUNING SÂU TỚI BÃO HÒA ---
        fine_dir.mkdir(parents=True, exist_ok=True)
        if not (tr_fine.stopped and (fine_dir / 'best_model.keras').is_file()):
            if tr_fine.next_epoch > 0 and (fine_dir / 'last_epoch.keras').is_file():
                print(f'🔄 Nối tiếp Pha Fine-tune từ Epoch {tr_fine.next_epoch+1} (LR={tr_fine.current_lr:.2e})...')
                model = compat_load_model(fine_dir / 'last_epoch.keras', compile=False)
            else:
                model = compat_load_model(head_dir / 'best_model.keras', compile=False)
            n_unf, n_tot = set_backbone_finetune_trainable(model, 'MobileNetV2', UNFREEZE_LAYERS)
            print(f'🔧 Fine-tune {n_unf}/{n_tot} tầng không phải BatchNorm (LR={tr_fine.current_lr:.2e}, max {MAX_EPOCHS_FINE} epochs)...')
            model.compile(optimizer=keras.optimizers.Adam(learning_rate=tr_fine.current_lr, clipnorm=1.0), loss=loss_fn, metrics=metrics_list)
            cb_f = ResumableSaturationCallback(fine_dir, tr_fine, local_run, CKPT_ARCHIVE_DIR, run_key, stage_num=2, sync_every=3)
            model.fit(train_ds, validation_data=val_ds, initial_epoch=tr_fine.next_epoch, epochs=MAX_EPOCHS_FINE, callbacks=[cb_f])
        else:
            print(f'✅ Pha Fine-tune đã bão hòa ở Epoch {tr_fine.best_epoch}/{tr_fine.next_epoch}.')

        best_model = compat_load_model(fine_dir / 'best_model.keras', compile=False)
        safe_save_keras(best_model, local_best_model)

    # --- ĐÁNH GIÁ & ĐÓNG GÓI KẾT QUẢ LÊN DRIVE ---
    h_best, h_stop = read_phase_epoch_stats(head_dir, tr_head)
    f_best, f_stop = read_phase_epoch_stats(fine_dir, tr_fine)
    sample_batch = next(iter(test_ds))[0]
    for _ in range(3): _ = best_model(sample_batch, training=False)
    tl0 = time.time()
    for _ in range(10): _ = best_model(sample_batch, training=False)
    latency_ms = ((time.time() - tl0) / 10.0 / BATCH_SIZE) * 1000.0

    res = evaluate_90_classes_complete(
        best_model, test_ds, test_raw, class_names, real_test_relpaths, local_eval_dir,
        extra_meta={
            'seed': seed,
            'model_key': 'mobilenet_v2',
            'model_title': 'MobileNetV2 (Từ số 0 + Bổ sung 8 loài)',
            'params_m': round(best_model.count_params() / 1e6, 2),
            'size_mb': round(local_best_model.stat().st_size / (1024*1024), 2),
            'latency_ms': round(latency_ms, 2),
            'head_best_epoch': h_best,
            'head_stopped_epoch': h_stop,
            'fine_best_epoch': f_best,
            'fine_stopped_epoch': f_stop,
            'train_session_min': round((time.time() - t0) / 60.0, 1),
        },
    )
    atomic_write_text(local_run / 'done.json', json.dumps({'seed': seed, 'status': 'completed'}, indent=2))
    save_dir_to_flat_zip(local_run, bundle_zip)
    try:
        (seed_dir / 'evaluation_report').mkdir(parents=True, exist_ok=True)
        safe_copy_file(local_best_model, drive_best_model)
        for fn in ('metrics.json', 'classification_report.csv', 'confusion_matrix.csv'):
            if (local_eval_dir / fn).is_file():
                safe_copy_file(local_eval_dir / fn, seed_dir / 'evaluation_report' / fn)
        for fn in ('class_names.json', f'split_counts_tu_so_0_seed_{seed}.csv', 'done.json'):
            if (local_run / fn).is_file():
                safe_copy_file(local_run / fn, seed_dir / fn)
    except OSError as exc:
        print(f'  ⚠️ Lưu bản sao rời gặp trễ FUSE ({exc}), đã lưu đủ trong {bundle_zip.name}')
    shutil.rmtree(split_dir, ignore_errors=True)
    ref = BASELINE_JSON['seeds_baseline'].get(str(seed), {})
    d_top1 = (res['top1_accuracy'] - ref.get('top1_accuracy', 0)) * 100.0
    print(f'🎉 HOÀN TẤT SEED {seed}: Top-1={res["top1_accuracy"]*100:.2f}% (Mẫu={ref.get("top1_accuracy",0)*100:.2f}%, Δ={d_top1:+.2f}%) | Top-3={res["top3_accuracy"]*100:.2f}% | Fine best/stop={f_best}/{f_stop}')
    return res


In [ ]:
# ==============================================================================
# === BƯỚC 4: CHẠY TỰ ĐỘNG TUẦN TỰ 3 SEEDS [99, 123, 999] & LƯU BÁO CÁO    ===
# ==============================================================================
def export_seed_comparison_reports() -> tuple[pd.DataFrame, pd.DataFrame]:
    seed_metrics, seed_reports = {}, {}
    for s in SEEDS_TO_RUN:
        m_obj, rep_df = read_eval_from_dir_or_bundle(
            SEEDS_OUT_ROOT / f'seed_{s}',
            COMPLETED_BUNDLES_DIR / f'seed_{s}_complete.zip',
        )
        if m_obj is not None and rep_df is not None:
            seed_metrics[s] = m_obj
            seed_reports[s] = rep_df
    df_comp = build_3seed_comparison_table(seed_metrics, BASELINE_JSON, SEEDS_TO_RUN)
    df_90 = build_90_species_comparison_table(seed_reports, BASELINE_JSON, SEEDS_TO_RUN)
    report_out = EXP_ROOT / 'bao_cao_doi_chieu_tu_so_0'
    report_out.mkdir(parents=True, exist_ok=True)
    if not df_comp.empty:
        safe_save_csv(df_comp, report_out / 'Doi_Chieu_3_Seeds_99_123_999_vs_Mau.csv', index=False, encoding='utf-8-sig')
    if not df_90.empty:
        safe_save_csv(df_90, report_out / 'Doi_Chieu_90_Loai_99_123_999_vs_Mau.csv', index=False, encoding='utf-8-sig')
    if not df_comp.empty and not df_90.empty:
        safe_save_excel({
            'Doi_Chieu_3_Seeds': (df_comp, False),
            'Chi_Tiet_90_Loai_vs_Mau': (df_90, False),
        }, report_out / 'Bao_Cao_3_Seeds_Tu_So_0_Doi_Chieu_Mau.xlsx')
    return df_comp, df_90

all_seed_results = {}
for s in SEEDS_TO_RUN:
    all_seed_results[s] = run_single_seed(s)
    export_seed_comparison_reports()


In [ ]:
# ==============================================================================
# === BƯỚC 5: HIỂN THỊ TỔNG HỢP MEAN ± STD, ĐỐI CHIẾU MẪU & 90 LOÀI CHI TIẾT ===
# ==============================================================================
df_comp, df_90 = export_seed_comparison_reports()
REPORT_OUT = EXP_ROOT / 'bao_cao_doi_chieu_tu_so_0'

print('\n=== ĐỐI CHIẾU TỔNG HỢP 3 SEEDS (99, 123, 999) vs BÁO CÁO MẪU ===')
display(df_comp)
print('\n=== CHI TIẾT 8 LOÀI BỔ SUNG THỰC TẾ TRONG 90 LOÀI ===')
display(df_90[df_90['Nhóm bổ sung'].str.contains('Bổ sung')])
print('\n=== TOP 10 LOÀI CẢI THIỆN F1 CAO NHẤT SO VỚI MẪU ===')
if 'Chênh lệch F1 (Mới - Mẫu)' in df_90.columns:
    display(df_90.sort_values('Chênh lệch F1 (Mới - Mẫu)', ascending=False).head(10))
print(f'\n✅ Đã xuất báo cáo tại: {REPORT_OUT}')
